# VerdaSense KB Ingestion — R4 Additional Embedding Models

**Purpose:** Build two new ChromaDB collections for the R4 embedding model
ablation, embedding the same 138 chunks from the 8 _kept.json sources using
two alternative models: BGE-large-en-v1.5 and E5-large-v2.

**Placement:** `rag-for-beginners/ingestion_R4_additional_models.py`
(project root — same level as `ingestion_full_8KB.ipynb`)

**Output directories (created by this notebook):**
```
rag-for-beginners/
  db_wound_care_v4/          ← MedEmbed (R4-A) — ALREADY BUILT, NOT TOUCHED
  db_wound_care_v4_bge/      ← BGE-large-en-v1.5  (R4-B) — built here
  db_wound_care_v4_e5/       ← E5-large-v2         (R4-C) — built here
```

**RTX 3050Ti (4 GB VRAM) safety strategy:**
- Models are loaded one at a time.
- Before loading the next model, the previous model's tensors are explicitly
  deleted and `torch.cuda.empty_cache()` is called.
- Each ingestion block is self-contained: load → ingest → verify → offload.
- BGE and E5 are each ~1.3 GB on GPU. With 4 GB VRAM, only one fits safely.
- ChromaDB itself runs on CPU; VRAM is used only for embedding computation.

**E5 prefix convention (mandatory):**
- E5-large-v2 requires `"passage: "` prefix on stored documents.
- At query time (in the R4 ablation script), queries use `"query: "` prefix.
- This is baked into the page_content of each document during E5 ingestion.
- MedEmbed and BGE: no prefix required.

**Do NOT re-run `ingestion_full_8KB.ipynb`.**
`db_wound_care_v4/` (MedEmbed) is complete and must not be overwritten.

**Cell structure:**
| Cell | Content |
|------|---------|
| Cell 0 | Imports & GPU safety helpers |
| Cell 1 | Paths & shared configuration |
| Cell 2 | GUIDELINE_METADATA (all 8 sources) |
| Cell 3 | WOUND_TYPE_MAP + WOUND_CATEGORY_MAP + population helper |
| Cell 4 | Load all 8 JSON files → `all_chunks` |
| Cell 5 | `chunks_to_documents()` converter |
| Cell 6 | Shared `ingest_to_chroma()` helper |
| Cell 7 | **BGE-large-en-v1.5** — load → ingest → verify → offload |
| Cell 8 | **E5-large-v2** — load (with passage: prefix) → ingest → verify → offload |
| Cell 9 | Final cross-verification (both DBs, chunk counts, smoke tests) |

## Cell 0 — Imports & GPU Safety Helpers

In [1]:
import os
import gc
import json
import shutil
import torch
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document as LC_Doc

# ── GPU safety helpers ────────────────────────────────────────────────────────

def report_vram(label: str = "") -> None:
    """Print current VRAM usage. Safe to call even without a GPU."""
    if torch.cuda.is_available():
        alloc  = torch.cuda.memory_allocated()  / 1024**3
        reserv = torch.cuda.memory_reserved()   / 1024**3
        total  = torch.cuda.get_device_properties(0).total_memory / 1024**3
        tag    = f"  [{label}] " if label else "  "
        print(f"{tag}VRAM: {alloc:.2f} GB allocated | {reserv:.2f} GB reserved | {total:.2f} GB total")
    else:
        print("  [no GPU] Running on CPU.")


def offload_embedding_model(model_obj, label: str = "") -> None:
    """
    Completely offload a HuggingFaceEmbeddings model from GPU memory.

    Steps:
      1. Move the underlying SentenceTransformer to CPU.
      2. Delete the Python reference.
      3. Collect Python garbage.
      4. Empty the CUDA cache.

    After this call, the model is fully unloaded and VRAM is reclaimed.
    """
    try:
        # SentenceTransformer lives inside .client
        if hasattr(model_obj, "client") and hasattr(model_obj.client, "to"):
            model_obj.client.to("cpu")
    except Exception as e:
        print(f"  [offload] CPU move warning: {e}")
    del model_obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  ✅ Model offloaded{' (' + label + ')' if label else ''}.")
    report_vram("post-offload")


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device        : {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
report_vram("startup")


c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device        : cuda
PyTorch version: 2.11.0+cu126
  [startup] VRAM: 0.00 GB allocated | 0.00 GB reserved | 4.00 GB total


## Cell 1 — Paths & Shared Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# This notebook lives at rag-for-beginners/ (project root).
# Adjust CHUNK_DIR if your _kept.json files are in a subfolder.

PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
CHUNK_DIR    = PROJECT_ROOT / "ingestion_output_ai"   # folder with all 8 _kept.json

# ── Output DB directories (never touch db_wound_care_v4 — MedEmbed) ──────────
DB_MEDEMBED  = PROJECT_ROOT / "db_wound_care_v4"       # R4-A — ALREADY DONE
DB_BGE       = PROJECT_ROOT / "db_wound_care_v4_bge"   # R4-B — built in Cell 7
DB_E5        = PROJECT_ROOT / "db_wound_care_v4_e5"    # R4-C — built in Cell 8

# ── ChromaDB settings ─────────────────────────────────────────────────────────
COLLECTION_NAME_BGE = "wound_care_v4_bge"
COLLECTION_NAME_E5  = "wound_care_v4_e5"
BATCH_SIZE          = 32

# ── Model names ───────────────────────────────────────────────────────────────
MODEL_BGE = "BAAI/bge-large-en-v1.5"
MODEL_E5  = "intfloat/e5-large-v2"

# ── _kept.json file map ───────────────────────────────────────────────────────
CHUNK_JSON_FILES = {
    "GP":    "GP_wound_dressings_kept.json",
    "SFP":   "SFP_wound_dressings_kept.json",
    "AJGP":  "AJGP_wound_dressings_kept.json",
    "WCM":   "WCM_wound_care_manual_kept.json",
    "EWMA":  "EWMA_wound_bed_preparation_kept.json",
    "ISTAP": "ISTAP_skin_tear_kept.json",
    "ANZBA": "ANZBA_burns_kept.json",
    "RCH":   "RCH_wound_care_kept.json",
}

print(f"Project root  : {PROJECT_ROOT}")
print(f"Chunk dir     : {CHUNK_DIR}")
print(f"DB MedEmbed   : {DB_MEDEMBED}  ← DO NOT OVERWRITE")
print(f"DB BGE        : {DB_BGE}")
print(f"DB E5         : {DB_E5}")
print(f"Batch size    : {BATCH_SIZE}")

# Safety check — confirm MedEmbed DB exists before proceeding
assert DB_MEDEMBED.exists(), (
    f"db_wound_care_v4 not found at {DB_MEDEMBED}. "
    "Run ingestion_full_8KB.ipynb first."
)
print(f"\n  ✅ db_wound_care_v4 confirmed present — will not be touched.")

Project root  : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners
Chunk dir     : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\ingestion_output_ai
DB MedEmbed   : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4  ← DO NOT OVERWRITE
DB BGE        : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_bge
DB E5         : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_e5
Batch size    : 32

  ✅ db_wound_care_v4 confirmed present — will not be touched.


## Cell 2 — GUIDELINE_METADATA (all 8 sources)

In [3]:
GUIDELINE_METADATA = {
    "GP_wound_dressings": {
        "authority":      "Ministry of Health Malaysia (MOH)",
        "year":           "2019",
        "guideline_type": "national_guideline",
        "full_name":      "Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer",
        "abbreviation":   "GP",
    },
    "WCM": {
        "authority":      "Ministry of Health Malaysia (MOH)",
        "year":           "2014",
        "guideline_type": "clinical_manual",
        "full_name":      "Wound Care Manual - First Edition",
        "abbreviation":   "WCM",
    },
    "AJGP": {
        "authority":      "Australian Journal of General Practice",
        "year":           "2022",
        "guideline_type": "clinical_review",
        "full_name":      "The art and science of selecting appropriate dressings for acute open wounds in general practice",
        "abbreviation":   "AJGP",
    },
    "SFP": {
        "authority":      "Singapore Family Physician",
        "year":           "2018",
        "guideline_type": "clinical_education",
        "full_name":      "Wound Dressings - A Primer For The Family Physician",
        "abbreviation":   "SFP",
    },
    "EWMA": {
        "authority":      "European Wound Management Association (EWMA)",
        "year":           "2004",
        "guideline_type": "international_evidence_based_position_document",
        "full_name":      "Wound Bed Preparation In Practice",
        "abbreviation":   "EWMA",
    },
    "ISTAP": {
        "authority":      "International Skin Tear Advisory Panel (ISTAP)",
        "year":           "2024",
        "guideline_type": "international_consensus_guideline",
        "full_name":      "Skin Tear Classification and Management (ISTAP)",
        "abbreviation":   "ISTAP",
    },
    "ANZBA": {
        "authority":      "Australia and New Zealand Burn Association (ANZBA)",
        "year":           "2021",
        "guideline_type": "national_specialist_guideline",
        "full_name":      "Burns Assessment and Management Guidelines (ANZBA)",
        "abbreviation":   "ANZBA",
    },
    "RCH": {
        "authority":      "Royal Children's Hospital Melbourne (RCH)",
        "year":           "2023",
        "guideline_type": "paediatric_clinical_practice_guideline",
        "full_name":      "Wound Assessment and Management — RCH Nursing Guideline (2023)",
        "abbreviation":   "RCH",
    },
}


def _resolve_guideline_meta(chunk: dict) -> dict:
    src = chunk.get("source", "")
    for key, meta in GUIDELINE_METADATA.items():
        if key.lower() in src.lower():
            return meta
    for key, meta in GUIDELINE_METADATA.items():
        if meta["abbreviation"].lower() in src.lower():
            return meta
    return {}


print("GUIDELINE_METADATA defined — 8 sources.")

GUIDELINE_METADATA defined — 8 sources.


## Cell 3 — WOUND_TYPE_MAP + WOUND_CATEGORY_MAP + Population Helper

Identical to `ingestion_full_8KB.ipynb` Cell 4. Copied verbatim — do not edit.

In [4]:
WOUND_TYPE_MAP: dict[str, str] = {

    # ── GP: algorithm + wound types 1–8 ──────────────────────────────────────
    "8409dedeea26": "general",    # GP TIME Framework
    "bd2bb8e1321e": "general",    # GP Decision Algorithm  ← the pinned algo chunk
    "52ef696853c7": "1",          # GP Wound Type 1
    "4643f10b8894": "2",          # GP Wound Type 2
    "c0a350e36ecf": "3",          # GP Wound Type 3
    "d622ee9f4c9c": "4",          # GP Wound Type 4
    "aad7a40107b0": "5",          # GP Wound Type 5
    "b4ba04cb08d4": "6",          # GP Wound Type 6
    "c4177e98524e": "7",          # GP Wound Type 7
    "e75347f9bdb3": "8",          # GP Wound Type 8
    "ca7a1e934891": "general",    # GP Referral criteria
    "8bb5168bd7e5": "general",    # GP Tissue type descriptions
    "0cc16688a29a": "general",    # GP Clinical glossary

    # ── SFP (36 chunks) ───────────────────────────────────────────────────────
    "ba05f42e79d0": "general",    # Abstract
    "08021aac2fa6": "general",    # Wound Dressings Selection Factors
    "e7ca1b602a48": "general",    # Wound Dressings Selection Factors
    "18ab673003dc": "general",    # Wound Dressings Selection Factors
    "07da3fcbedeb": "general",    # Categories of Wound Dressings overview
    "14535d1dba1a": "general",    # Moisture Retentive Dressings
    "8443058d681e": "general",    # Absorbent Dressings
    "c42103c45a0c": "general",    # Absorbent Dressings
    "80e829def300": "general",    # Antimicrobial Dressings
    "3082e1a296e7": "general",    # Antimicrobial Dressings (iodine/silver)
    "5972b0ef4b66": "general",    # Composite Dressings
    "b49bc7f134bd": "general",    # Protective Dressings
    "08bd7cc4c34e": "general",    # Advances in Wound Care Technology
    "9c6c8076a17e": "procedure",  # Maggot Debridement Therapy
    "138be9123963": "general",    # Growth Factors
    "218879320005": "general",    # Growth Factors
    "52981701d491": "general",    # Growth Factors
    "8b8a70dc7e9e": "general",    # Bioengineered Skin Substitutes
    "ea9194262a27": "procedure",  # NPWT
    "1fddeefdfb8b": "procedure",    # Oxygen Therapy HBOT
    "a23f76944381": "procedure",    # Ultrasound Therapy
    "352258a65d26": "procedure",    # Low-Power Laser Therapy
    "8ac10c306384": "general",    # Conclusions
    "25a2969b1f2e": "general",    # Learning Points
    "6a4054e5b255": "general",    # Learning Points
    "b4c13d77818b": "general",    # Table 2 Hydrocolloids
    "ad036dc35955": "general",    # Table 2 Hydrogels
    "3b666ccfba99": "general",    # Table 2 Alginates
    "4b75fefc0517": "general",    # Table 2 Hydrofiber
    "254dd74d7f00": "general",    # Table 2 Foams
    "f05456b64b8d": "general",    # Table 2 Cadexomer Iodine
    "765445bd6358": "general",    # Table 2 Silver Barrier
    "757123a126d6": "general",    # Table 2 Non-Adherent Synthetic
    "9e661711e520": "general",    # Table 2 Films/Membranes
    "605598290337": "general",    # Table 2 Gauze
    "e86bb5a0ee36": "general",    # Table 2 Composite Dressing

    # ── AJGP (19 chunks) ──────────────────────────────────────────────────────
    "65414974ff6f": "general",    # Background & Objective
    "e33a690e240a": "general",    # Introduction Acute Wound Context
    "2dc1f26b6233": "general",    # Wound Dressings General Principles
    "4b741bf1a512": "general",    # Wound Dressings General Principles
    "6e34c2b50050": "general",    # Description of Dressings Overview
    "55569f16010f": "skin_tear",  # Table 1 Skin tears
    "18fdcc9d34d9": "general",    # Table 1 Minor cut/laceration
    "6555b7728ccc": "general",    # Table 1 Postoperative wounds
    "5116098922da": "burn",       # Table 1 Small superficial burns
    "0d0a9fc09c73": "dfu",        # Table 1 Diabetic foot
    "f7b66b783d4d": "general",    # Dressing Types Film
    "c9b68759e2a8": "general",    # Dressing Types Foam
    "e5409b86f782": "general",    # Dressing Types Low-Adherent
    "a1968343e3ba": "general",    # Dressing Types Hydrocolloid
    "38f855618fe2": "general",    # Dressing Types Alginate
    "0c49aa8d13c4": "general",    # Dressing Types Antimicrobial
    "ca472c44f89b": "general",    # Dressing Types and Costs
    "751c661d8c3c": "general",    # Conclusion
    "10c6b59b1aff": "general",    # Key Points

    # ── WCM (40 chunks) ───────────────────────────────────────────────────────
    "0305af9f4828": "general",    # Ch1 Skin Anatomy
    "06f9b3a7dc69": "general",    # Ch2 Wound Definition & Classification
    "9f8aabde769e": "general",    # Ch3 Wound Assessment Principles
    "9eace8705738": "general",    # Ch4 Wound Infection Pathway
    "424af37cbc2e": "general",    # Ch4 Antibiotic Treatment
    "5c0355fe4e4c": "burn",       # Ch7a Burn Wound Assessment & Management
    "594eb6407bf0": "general",    # Ch7b Traumatic Wound
    "e21d5fa201a5": "dfu",        # Ch8a DFU Assessment & Wagner Classification
    "261f5855cdb7": "dfu",        # Ch8a DFU Management & Foot Care
    "4b89a41a249d": "vlu",        # Ch8b Venous Ulcer
    "90610861be44": "general",    # Ch8c Arterial Ulcer
    "cde2bed226ee": "general",    # Ch8d Pressure Ulcer
    "7ef21910c0c1": "general",    # Ch9 Non-Healing Ulcer
    "c6bd9ad78883": "general",    # Ch9 Non-Healing Ulcer
    "3ab46ddb1cb9": "general",    # Ch10 Necrotizing Fasciitis
    "634395d03254": "general",    # Ch10 Necrotizing Fasciitis
    "1101f1aec597": "procedure",  # Ch11 Pain Management
    "42616730845e": "procedure",  # Ch11 Pain Management
    "d1b94d1b0458": "procedure",  # Ch11 Pain Management
    "9177c6a0a15f": "procedure",  # Ch11 Pain Management
    "fd2fb7bfa667": "procedure",  # Ch11 Pain Management
    "fc529d1bd75b": "procedure",  # Ch13 Wound Cleansing Solutions
    "1b2820149c46": "general",    # Ch14 Dressing Purpose & Categories Overview
    "2de03f803f2f": "general",    # Ch14 Film
    "d81176511903": "general",    # Ch14 Hydrogel
    "f8cb463d04cf": "general",    # Ch14 Hydrocolloid
    "c540b3e5c067": "general",    # Ch14 Calcium Alginate
    "77e6e32d188a": "general",    # Ch14 Foams
    "e63bd0378895": "general",    # Ch14 Hydrofibre
    "861a57a2172c": "general",    # Ch14 Charcoal
    "e8c86c4e1aa6": "general",    # Ch14 Silver
    "6fd9e2433cc9": "general",    # Ch14 Polymeric Membrane
    "9466f8d45d86": "general",    # Ch14 Composite Dressing
    "ac13950bf23d": "general",    # Ch14 Other Advanced Dressings
    "b5b5a6c9dcf2": "procedure",  # Ch15 Wound Debridement Methods
    "c550f2c4e065": "procedure",  # Ch15 Wound Debridement Methods
    "c45690235ea6": "procedure",  # Ch15 Wound Debridement Methods
    "b480aa73a9c2": "general",    # Ch16a Honey Dressing
    "05cc6ca1ddfc": "procedure",  # Ch16c NPWT
    "c799dd10dcf3": "procedure",  # Appendix 7 Analgesics

    # ── EWMA (12 chunks) ──────────────────────────────────────────────────────
    "34a1a51c74ae": "general",    # TIME Framework — Evolution & Four Components
    "e905d7d38dad": "general",    # TIME Applied to Practice — Pathway & WBP
    "643cd131813b": "general",    # TIME Figure 1 — Dynamic Wound Progression
    "61346aa382bd": "dfu",        # DFU — Before TIME & Tissue Management
    "65a59d1430fd": "dfu",        # DFU — Inflammation & Infection Control
    "9525c0cb50e8": "dfu",        # DFU — Moisture Balance
    "6231fe39e6d5": "dfu",        # DFU — Edge Advancement
    "dfb568f55887": "dfu",        # DFU — Advanced Therapies & After TIME
    "142adfaa2033": "vlu",        # VLU — Before TIME & Tissue Management
    "bd87881f1796": "vlu",        # VLU — Inflammation & Infection Control
    "9d379b10e0c1": "vlu",        # VLU — Moisture Balance
    "a60e6a06f137": "vlu",        # VLU — Edge Advancement & Advanced Therapies

    # ── ISTAP (3 chunks) ──────────────────────────────────────────────────────
    "d2957f5e4841": "skin_tear",  # ISTAP Classification Types 1, 2, 3
    "c3d5e1f498ba": "skin_tear",  # ISTAP Pathway to Assessment and Treatment
    "3bde291790e7": "skin_tear",  # ISTAP Product Selection Guide

    # ── ANZBA (4 chunks) ──────────────────────────────────────────────────────
    "5d08501c9e7a": "burn",       # ANZBA Burn Classification, Depth & Initial Dressing
    "66d4e6fcfa13": "burn",       # ANZBA Referral Criteria
    "73bc744c2269": "burn",       # ANZBA First Aid, Hydrogel, Initial Wound Cover
    "1c099dac6b20": "burn",       # ANZBA Dressing Selection Reference

    # ── RCH (11 chunks) ───────────────────────────────────────────────────────
    "dd7540d2be3e": "paediatric", # TIME Wound Assessment Framework (RCH)
    "24994a09fcdd": "paediatric", # Wound Assessment Infection/Inflammation (RCH)
    "66b18503ea05": "paediatric", # Wound Assessment Moisture/Exudate (RCH)
    "7f0dde09a620": "paediatric", # Factors Affecting Wound Healing (RCH)
    "3d27931cee1d": "paediatric", # Wound Management Principles (RCH)
    "6cff871dda23": "paediatric", # Primary Dressing Selection Table (RCH)
    "69b3535fbd9a": "paediatric", # Secondary Dressing Selection (RCH)
    "dc0189fecaf3": "paediatric", # Acute Traumatic Wound × Dressing Table (RCH)
    "0b0626e45180": "paediatric", # Dressing Choices Full Detail Table (RCH)
    "d2e54ae32ac9": "paediatric", # RCH Product Reference Guide
    "b4b0b902c29a": "paediatric", # RCH Dressings Poster
}

WOUND_CATEGORY_MAP: dict[str, str] = {

    # ── GP ────────────────────────────────────────────────────────────────────
    "8409dedeea26": "assessment",          # GP TIME Framework
    "bd2bb8e1321e": "algorithm",           # GP Decision Algorithm ← primary pinned
    "52ef696853c7": "algorithm",           # GP Wound Type 1
    "4643f10b8894": "algorithm",           # GP Wound Type 2
    "c0a350e36ecf": "algorithm",           # GP Wound Type 3
    "d622ee9f4c9c": "algorithm",           # GP Wound Type 4
    "aad7a40107b0": "algorithm",           # GP Wound Type 5
    "b4ba04cb08d4": "algorithm",           # GP Wound Type 6
    "c4177e98524e": "algorithm",           # GP Wound Type 7
    "e75347f9bdb3": "algorithm",           # GP Wound Type 8
    "ca7a1e934891": "assessment",          # GP Referral criteria
    "8bb5168bd7e5": "assessment",          # GP Tissue type descriptions
    "0cc16688a29a": "reference",           # GP Clinical glossary

    # ── SFP ───────────────────────────────────────────────────────────────────
    "ba05f42e79d0": "reference",
    "08021aac2fa6": "dressing_mechanism",
    "e7ca1b602a48": "dressing_mechanism",
    "18ab673003dc": "dressing_mechanism",
    "07da3fcbedeb": "dressing_mechanism",
    "14535d1dba1a": "dressing_product",
    "8443058d681e": "dressing_product",
    "c42103c45a0c": "dressing_product",
    "80e829def300": "dressing_product",
    "3082e1a296e7": "dressing_product",    # iodine/silver/honey — thyroid contraindication
    "5972b0ef4b66": "dressing_product",
    "b49bc7f134bd": "dressing_product",
    "08bd7cc4c34e": "procedure",
    "9c6c8076a17e": "procedure",
    "138be9123963": "procedure",
    "218879320005": "procedure",
    "52981701d491": "procedure",
    "8b8a70dc7e9e": "procedure",
    "ea9194262a27": "procedure",
    "1fddeefdfb8b": "procedure",
    "a23f76944381": "procedure",
    "352258a65d26": "procedure",
    "8ac10c306384": "reference",
    "25a2969b1f2e": "reference",
    "6a4054e5b255": "reference",
    "b4c13d77818b": "dressing_product",    # Table 2 Hydrocolloids
    "ad036dc35955": "dressing_product",    # Table 2 Hydrogels
    "3b666ccfba99": "dressing_product",    # Table 2 Alginates
    "4b75fefc0517": "dressing_product",    # Table 2 Hydrofiber
    "254dd74d7f00": "dressing_product",    # Table 2 Foams
    "f05456b64b8d": "dressing_product",    # Table 2 Cadexomer Iodine
    "765445bd6358": "dressing_product",    # Table 2 Silver Barrier
    "757123a126d6": "dressing_product",    # Table 2 Non-Adherent
    "9e661711e520": "dressing_product",    # Table 2 Films
    "605598290337": "dressing_product",    # Table 2 Gauze
    "e86bb5a0ee36": "dressing_product",    # Table 2 Composite

    # ── AJGP ──────────────────────────────────────────────────────────────────
    "65414974ff6f": "reference",
    "e33a690e240a": "assessment",
    "2dc1f26b6233": "dressing_mechanism",
    "4b741bf1a512": "dressing_mechanism",
    "6e34c2b50050": "dressing_mechanism",
    "55569f16010f": "wound_specific",      # skin tear table
    "18fdcc9d34d9": "wound_specific",      # minor cut/laceration table
    "6555b7728ccc": "wound_specific",      # postoperative wounds table
    "5116098922da": "wound_specific",      # burns table
    "0d0a9fc09c73": "wound_specific",      # diabetic foot table
    "f7b66b783d4d": "dressing_product",    # Film
    "c9b68759e2a8": "dressing_product",    # Foam
    "e5409b86f782": "dressing_product",    # Low-adherent
    "a1968343e3ba": "dressing_product",    # Hydrocolloid
    "38f855618fe2": "dressing_product",    # Alginate
    "0c49aa8d13c4": "dressing_product",    # Antimicrobial
    "ca472c44f89b": "reference",
    "751c661d8c3c": "reference",
    "10c6b59b1aff": "reference",

    # ── WCM ───────────────────────────────────────────────────────────────────
    "0305af9f4828": "assessment",
    "06f9b3a7dc69": "assessment",
    "9f8aabde769e": "assessment",
    "9eace8705738": "assessment",
    "424af37cbc2e": "assessment",
    "5c0355fe4e4c": "wound_specific",      # Burn chapter
    "594eb6407bf0": "wound_specific",      # Traumatic wound
    "e21d5fa201a5": "wound_specific",      # DFU assessment
    "261f5855cdb7": "wound_specific",      # DFU management
    "4b89a41a249d": "wound_specific",      # Venous ulcer
    "90610861be44": "wound_specific",      # Arterial ulcer
    "cde2bed226ee": "wound_specific",      # Pressure ulcer
    "7ef21910c0c1": "wound_specific",      # Non-healing ulcer
    "c6bd9ad78883": "wound_specific",
    "3ab46ddb1cb9": "wound_specific",      # Necrotizing fasciitis
    "634395d03254": "wound_specific",
    "1101f1aec597": "procedure",
    "42616730845e": "procedure",
    "d1b94d1b0458": "procedure",
    "9177c6a0a15f": "procedure",
    "fd2fb7bfa667": "procedure",
    "fc529d1bd75b": "procedure",           # Ch13 Cleansing
    "1b2820149c46": "dressing_mechanism",  # Ch14 Overview
    "2de03f803f2f": "dressing_product",    # Ch14 Film
    "d81176511903": "dressing_product",    # Ch14 Hydrogel
    "f8cb463d04cf": "dressing_product",    # Ch14 Hydrocolloid
    "c540b3e5c067": "dressing_product",    # Ch14 Alginate
    "77e6e32d188a": "dressing_product",    # Ch14 Foam
    "e63bd0378895": "dressing_product",    # Ch14 Hydrofibre
    "861a57a2172c": "dressing_product",    # Ch14 Charcoal
    "e8c86c4e1aa6": "dressing_product",    # Ch14 Silver
    "6fd9e2433cc9": "dressing_product",    # Ch14 Polymeric Membrane
    "9466f8d45d86": "dressing_product",    # Ch14 Composite
    "ac13950bf23d": "dressing_product",    # Ch14 Other Advanced
    "b5b5a6c9dcf2": "procedure",           # Ch15 Debridement
    "c550f2c4e065": "procedure",
    "c45690235ea6": "procedure",
    "b480aa73a9c2": "dressing_product",    # Ch16a Honey
    "05cc6ca1ddfc": "procedure",           # Ch16c NPWT
    "c799dd10dcf3": "reference",           # Appendix 7 Analgesics

    # ── EWMA ──────────────────────────────────────────────────────────────────
    "34a1a51c74ae": "assessment",          # TIME Framework evolution
    "e905d7d38dad": "algorithm",           # TIME Applied to Practice — second TIME algo source
    "643cd131813b": "assessment",          # TIME Figure 1
    "61346aa382bd": "wound_specific",      # DFU Tissue Management
    "65a59d1430fd": "wound_specific",      # DFU Infection Control
    "9525c0cb50e8": "wound_specific",      # DFU Moisture Balance
    "6231fe39e6d5": "wound_specific",      # DFU Edge Advancement
    "dfb568f55887": "wound_specific",      # DFU Advanced Therapies
    "142adfaa2033": "wound_specific",      # VLU Tissue Management
    "bd87881f1796": "wound_specific",      # VLU Infection Control
    "9d379b10e0c1": "wound_specific",      # VLU Moisture Balance
    "a60e6a06f137": "wound_specific",      # VLU Edge Advancement & Advanced

    # ── ISTAP ─────────────────────────────────────────────────────────────────
    "d2957f5e4841": "assessment",          # ISTAP Classification Types 1, 2, 3
    "c3d5e1f498ba": "algorithm",           # ISTAP Pathway (treatment algorithm)
    "3bde291790e7": "dressing_product",    # ISTAP Product Selection

    # ── ANZBA ─────────────────────────────────────────────────────────────────
    "5d08501c9e7a": "assessment",          # ANZBA Burn Classification & Depth
    "66d4e6fcfa13": "algorithm",           # ANZBA Referral Criteria (algorithmic)
    "73bc744c2269": "procedure",           # ANZBA First Aid
    "1c099dac6b20": "dressing_product",    # ANZBA Dressing Selection Reference

    # ── RCH ───────────────────────────────────────────────────────────────────
    "dd7540d2be3e": "assessment",          # TIME Wound Assessment (RCH)
    "24994a09fcdd": "assessment",          # Infection/Inflammation Assessment (RCH)
    "66b18503ea05": "assessment",          # Moisture/Exudate Assessment (RCH)
    "7f0dde09a620": "assessment",          # Factors Affecting Wound Healing (RCH)
    "3d27931cee1d": "procedure",           # Wound Management Principles (RCH)
    "6cff871dda23": "dressing_product",  # Primary Dressing Selection Table (RCH)
    "69b3535fbd9a": "dressing_product",    # Secondary Dressing Selection (RCH)
    "dc0189fecaf3": "dressing_product",      # Acute Traumatic Wound × Dressing (RCH)
    "0b0626e45180": "dressing_product",    # Dressing Choices Full Detail (RCH)
    "d2e54ae32ac9": "dressing_product",           # Product Reference Guide (RCH)
    "b4b0b902c29a": "dressing_product",    # Dressings Poster (RCH)
}


def _resolve_population(chunk: dict) -> str:
    return chunk.get("population", "adult")


print(f"WOUND_TYPE_MAP     : {len(WOUND_TYPE_MAP)} entries")
print(f"WOUND_CATEGORY_MAP : {len(WOUND_CATEGORY_MAP)} entries")


WOUND_TYPE_MAP     : 138 entries
WOUND_CATEGORY_MAP : 138 entries


## Cell 4 — Load All 8 JSON Files → `all_chunks`

Identical to `ingestion_full_8KB.ipynb` Cell 5. The chunks are the same
for all embedding models — only the vectors change.

In [5]:
def load_all_chunks(chunk_dir: Path) -> list[dict]:
    all_chunks: list[dict] = []
    for src_key, fname in CHUNK_JSON_FILES.items():
        fpath = chunk_dir / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"Missing: {fpath}\n"
                f"Set CHUNK_DIR to the folder containing all 8 _kept.json files."
            )
        raw = json.load(open(fpath, encoding="utf-8"))
        chunks = raw if isinstance(raw, list) else raw.get("kept_chunks", [])
        print(f"  ✅ [{src_key:<5}] {len(chunks):>3} chunks  ←  {fname}")
        all_chunks.extend(chunks)
    print(f"\n  Total: {len(all_chunks)} chunks across {len(CHUNK_JSON_FILES)} sources")
    return all_chunks


print("[Cell 4] Loading all chunk JSON files...")
all_chunks = load_all_chunks(CHUNK_DIR)

[Cell 4] Loading all chunk JSON files...
  ✅ [GP   ]  13 chunks  ←  GP_wound_dressings_kept.json
  ✅ [SFP  ]  36 chunks  ←  SFP_wound_dressings_kept.json
  ✅ [AJGP ]  19 chunks  ←  AJGP_wound_dressings_kept.json
  ✅ [WCM  ]  40 chunks  ←  WCM_wound_care_manual_kept.json
  ✅ [EWMA ]  12 chunks  ←  EWMA_wound_bed_preparation_kept.json
  ✅ [ISTAP]   3 chunks  ←  ISTAP_skin_tear_kept.json
  ✅ [ANZBA]   4 chunks  ←  ANZBA_burns_kept.json
  ✅ [RCH  ]  11 chunks  ←  RCH_wound_care_kept.json

  Total: 138 chunks across 8 sources


## Cell 5 — `chunks_to_documents()` Converter

Identical to `ingestion_full_8KB.ipynb` Cell 6, with one addition:
an optional `page_content_prefix` parameter used by E5 ingestion
to prepend `"passage: "` to every document's page_content.

In [6]:
def chunks_to_documents(
    chunks: list[dict],
    page_content_prefix: str = "",
) -> list[LC_Doc]:
    """
    Convert raw chunk dicts to LangChain Document objects.

    page_content = page_content_prefix + ai_summary
    metadata     = all standard fields (identical to ingestion_full_8KB.ipynb)

    Parameters
    ----------
    chunks               : list of raw chunk dicts from _kept.json files
    page_content_prefix  : string prepended to ai_summary before embedding.
                           ""         for MedEmbed (R4-A) and BGE (R4-B)
                           "passage: " for E5-large-v2 (R4-C) — mandatory per model card
    """
    docs: list[LC_Doc] = []
    missing_summary = []

    for chunk in chunks:
        chunk_id = chunk.get("chunk_id", "UNKNOWN")
        summary  = chunk.get("ai_summary", "").strip()

        if not summary:
            missing_summary.append(chunk_id)
            summary = chunk.get("section", chunk.get("title", f"Chunk {chunk_id}"))

        g_meta         = _resolve_guideline_meta(chunk)
        wound_type     = WOUND_TYPE_MAP.get(chunk_id, "general")
        wound_category = WOUND_CATEGORY_MAP.get(chunk_id, "general")
        population     = _resolve_population(chunk)

        metadata = {
            "chunk_id":       chunk_id,
            "source":         str(chunk.get("source", "")),
            "section":        str(chunk.get("section", "")),
            "parent_section": str(chunk.get("parent_section", "")),
            "chunk_index":    int(chunk.get("chunk_index", 0)),
            "char_count":     int(chunk.get("char_count", len(summary))),
            "raw_text":       str(chunk.get("text", summary)),
            "wound_type":     wound_type,
            "wound_category": wound_category,
            "population":     population,
            "authority":      str(g_meta.get("authority",      "")),
            "year":           str(g_meta.get("year",           "")),
            "guideline_type": str(g_meta.get("guideline_type", "")),
            "full_name":      str(g_meta.get("full_name",      "")),
            "abbreviation":   str(g_meta.get("abbreviation",   "")),
        }
        if "source_collection" in chunk:
            metadata["source_collection"] = str(chunk["source_collection"])

        page_content = page_content_prefix + summary
        docs.append(LC_Doc(page_content=page_content, metadata=metadata))

    if missing_summary:
        print(f"  ⚠️  {len(missing_summary)} chunks used section as ai_summary fallback: {missing_summary[:5]}")

    return docs


print("chunks_to_documents() defined.")
print("  Prefix modes:")
print('    MedEmbed / BGE : page_content_prefix=""       (no prefix)')
print('    E5-large-v2    : page_content_prefix="passage: " (mandatory)')

chunks_to_documents() defined.
  Prefix modes:
    MedEmbed / BGE : page_content_prefix=""       (no prefix)
    E5-large-v2    : page_content_prefix="passage: " (mandatory)


## Cell 6 — Shared `ingest_to_chroma()` Helper

In [7]:
def ingest_to_chroma(
    documents:       list[LC_Doc],
    db_dir:          Path,
    embed_model:     HuggingFaceEmbeddings,
    collection_name: str,
    batch_size:      int = BATCH_SIZE,
    overwrite:       bool = True,
) -> Chroma:
    """
    Ingest LangChain Documents into a ChromaDB directory.

    overwrite=True: wipe db_dir first (safe — it is a new directory for BGE/E5).
    """
    if overwrite and db_dir.exists():
        print(f"  Removing existing directory: {db_dir}")
        shutil.rmtree(db_dir)

    db = Chroma(
        persist_directory   = str(db_dir),
        embedding_function  = embed_model,
        collection_name     = collection_name,
        collection_metadata = {"hnsw:space": "cosine"},
    )

    total_batches = (len(documents) + batch_size - 1) // batch_size
    print(f"  Ingesting {len(documents)} docs in {total_batches} batches of {batch_size}...")

    for batch_num, start in enumerate(range(0, len(documents), batch_size), 1):
        batch = documents[start : start + batch_size]
        ids   = [doc.metadata["chunk_id"] for doc in batch]

        dupes = [cid for cid, cnt in
                 {i: ids.count(i) for i in ids}.items() if cnt > 1]
        if dupes:
            raise ValueError(f"Duplicate chunk_ids in batch {batch_num}: {set(dupes)}")

        db.add_documents(documents=batch, ids=ids)
        print(f"  Batch {batch_num:>2}/{total_batches}: {len(batch):>3} docs  "
              f"[{ids[0]} … {ids[-1]}]")

    print(f"\n  ✅ Ingestion complete — {len(documents)} documents in {db_dir}")
    return db


def verify_db(db: Chroma, db_dir: Path, expected_count: int = 138) -> None:
    """Run verification checks and smoke tests on a newly built DB."""
    raw   = db.get(include=["metadatas", "documents"])
    n     = len(raw["ids"])
    metas = raw["metadatas"]

    print(f"\n  Total docs   : {n}  (expected {expected_count})")
    assert n == expected_count, f"⚠ Count mismatch: got {n}, expected {expected_count}"
    print(f"  ✅ Count verified: {n}")

    # Source breakdown
    src_counts: dict[str, int] = {}
    for m in metas:
        abbr = m.get("abbreviation", m.get("source", "?"))
        src_counts[abbr] = src_counts.get(abbr, 0) + 1
    print("\n  Docs by source:")
    for abbr, cnt in sorted(src_counts.items()):
        print(f"    {abbr:<8}: {cnt:>3}")

    # wound_category breakdown
    wcat_counts: dict[str, int] = {}
    for m in metas:
        wcat_counts[m.get("wound_category","?")] = wcat_counts.get(m.get("wound_category","?"),0)+1
    print("\n  Docs by wound_category:")
    for wc, cnt in sorted(wcat_counts.items()):
        print(f"    {wc:<22}: {cnt:>3}")

    # Smoke test
    print("\n  Smoke test — 'foam dressing high exudate wound care':")
    res = db.similarity_search("foam dressing high exudate wound care", k=3)
    for i, r in enumerate(res, 1):
        print(f"    [{i}] {r.metadata.get('abbreviation','?'):<6} | "
              f"wtype={r.metadata.get('wound_type','?'):<12} | "
              f"{r.page_content[:80]}...")

    print("\n  Smoke test — 'antimicrobial silver dressing infected wound':")
    res2 = db.similarity_search("antimicrobial silver dressing infected wound", k=3)
    for i, r in enumerate(res2, 1):
        print(f"    [{i}] {r.metadata.get('abbreviation','?'):<6} | "
              f"wtype={r.metadata.get('wound_type','?'):<12} | "
              f"{r.page_content[:80]}...")


print("ingest_to_chroma() and verify_db() defined.")

ingest_to_chroma() and verify_db() defined.


## Cell 7 — BGE-large-en-v1.5: Load → Ingest → Verify → Offload

**RTX 3050Ti notes:**
- BGE-large-en-v1.5 weights: ~1.34 GB on GPU (fp32). Fits in 4 GB VRAM.
- No query prefix required for BGE-large-en-v1.5 (the -v1.5 update removed
  the mandatory instruction prefix for retrieval tasks). Page content stored
  as-is (same format as MedEmbed).
- After ingestion, the model is fully offloaded before E5 is loaded.

In [8]:
print("=" * 65)
print("  BLOCK 1 — BGE-large-en-v1.5  →  db_wound_care_v4_bge/")
print("=" * 65)
report_vram("before BGE load")

# ── Step 1: Load BGE embedding model ─────────────────────────────────────────
print(f"\n[7.1] Loading {MODEL_BGE} ...")
embed_bge = HuggingFaceEmbeddings(
    model_name   = MODEL_BGE,
    model_kwargs = {"device": DEVICE},
    encode_kwargs= {"normalize_embeddings": True},
)
print("  ✅ BGE model loaded.")
report_vram("after BGE load")

# ── Step 2: Convert chunks to documents (no prefix for BGE) ──────────────────
print("\n[7.2] Converting chunks to Documents (no prefix)...")
docs_bge = chunks_to_documents(all_chunks, page_content_prefix="")
print(f"  ✅ {len(docs_bge)} Documents ready.")

# ── Step 3: Ingest into db_wound_care_v4_bge ─────────────────────────────────
print(f"\n[7.3] Ingesting into {DB_BGE} ...")
db_bge = ingest_to_chroma(
    documents       = docs_bge,
    db_dir          = DB_BGE,
    embed_model     = embed_bge,
    collection_name = COLLECTION_NAME_BGE,
    overwrite       = True,
)

# ── Step 4: Verify ────────────────────────────────────────────────────────────
print("\n[7.4] Verifying db_wound_care_v4_bge ...")
verify_db(db_bge, DB_BGE, expected_count=138)
del db_bge   # release Chroma handle

# ── Step 5: Offload BGE from GPU ──────────────────────────────────────────────
print("\n[7.5] Offloading BGE model from VRAM ...")
offload_embedding_model(embed_bge, label="BGE-large-en-v1.5")

print("\n  ✅ BLOCK 1 COMPLETE — db_wound_care_v4_bge/ built and verified.")
print("  BGE model fully offloaded. VRAM reclaimed for E5.")

  BLOCK 1 — BGE-large-en-v1.5  →  db_wound_care_v4_bge/
  [before BGE load] VRAM: 0.00 GB allocated | 0.00 GB reserved | 4.00 GB total

[7.1] Loading BAAI/bge-large-en-v1.5 ...


c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GIGA\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5044.60it/s]
Be

  ✅ BGE model loaded.
  [after BGE load] VRAM: 1.25 GB allocated | 1.26 GB reserved | 4.00 GB total

[7.2] Converting chunks to Documents (no prefix)...
  ✅ 138 Documents ready.

[7.3] Ingesting into c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_bge ...
  Ingesting 138 docs in 5 batches of 32...
  Batch  1/5:  32 docs  [8409dedeea26 … ea9194262a27]
  Batch  2/5:  32 docs  [1fddeefdfb8b … 38f855618fe2]
  Batch  3/5:  32 docs  [0c49aa8d13c4 … 77e6e32d188a]
  Batch  4/5:  32 docs  [e63bd0378895 … dd7540d2be3e]
  Batch  5/5:  10 docs  [24994a09fcdd … b4b0b902c29a]

  ✅ Ingestion complete — 138 documents in c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_bge

[7.4] Verifying db_wound_care_v4_bge ...

  Total docs   : 138  (expected 138)
  ✅ Count verified: 138

  Docs by source:
            : 108
    ANZBA   :   4
    EWMA    :  12
    ISTAP   :   3
    RCH     :  11

  Docs by wound_category:
    algorithm   

## Cell 8 — E5-large-v2: Load → Ingest (with passage: prefix) → Verify → Offload

**RTX 3050Ti notes:**
- E5-large-v2 weights: ~1.34 GB on GPU (fp32). Fits in 4 GB VRAM.
- **Mandatory prefix:** E5-large-v2 requires `"passage: "` prefix on stored
  documents and `"query: "` prefix on query strings at retrieval time.
  This is a documented requirement from the E5 paper (Wang et al., 2022).
  Skipping the prefix degrades E5 retrieval quality by 2–5 pp on BEIR benchmarks.
- The `"passage: "` prefix is baked into page_content here at ingestion time.
- The `"query: "` prefix is applied in the R4 ablation script at search time.
- After ingestion, E5 is offloaded.

In [9]:
print("=" * 65)
print("  BLOCK 2 — E5-large-v2  →  db_wound_care_v4_e5/")
print("=" * 65)
report_vram("before E5 load")

# ── Step 1: Load E5 embedding model ──────────────────────────────────────────
print(f"\n[8.1] Loading {MODEL_E5} ...")
embed_e5 = HuggingFaceEmbeddings(
    model_name   = MODEL_E5,
    model_kwargs = {"device": DEVICE},
    encode_kwargs= {"normalize_embeddings": True},
)
print("  ✅ E5 model loaded.")
report_vram("after E5 load")

# ── Step 2: Convert chunks with "passage: " prefix ───────────────────────────
print('\n[8.2] Converting chunks to Documents (prefix="passage: ") ...')
docs_e5 = chunks_to_documents(all_chunks, page_content_prefix="passage: ")
print(f"  ✅ {len(docs_e5)} Documents ready.")
# Confirm prefix applied
print(f"  Sample page_content prefix check: '{docs_e5[0].page_content[:20]}...'")

# ── Step 3: Ingest into db_wound_care_v4_e5 ──────────────────────────────────
print(f"\n[8.3] Ingesting into {DB_E5} ...")
db_e5 = ingest_to_chroma(
    documents       = docs_e5,
    db_dir          = DB_E5,
    embed_model     = embed_e5,
    collection_name = COLLECTION_NAME_E5,
    overwrite       = True,
)

# ── Step 4: Verify ────────────────────────────────────────────────────────────
print("\n[8.4] Verifying db_wound_care_v4_e5 ...")
verify_db(db_e5, DB_E5, expected_count=138)
del db_e5   # release Chroma handle

# ── Step 5: Offload E5 from GPU ───────────────────────────────────────────────
print("\n[8.5] Offloading E5 model from VRAM ...")
offload_embedding_model(embed_e5, label="E5-large-v2")

print("\n  ✅ BLOCK 2 COMPLETE — db_wound_care_v4_e5/ built and verified.")
print("  E5 model fully offloaded.")

  BLOCK 2 — E5-large-v2  →  db_wound_care_v4_e5/
  [before E5 load] VRAM: 1.26 GB allocated | 1.32 GB reserved | 4.00 GB total

[8.1] Loading intfloat/e5-large-v2 ...


c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GIGA\.cache\huggingface\hub\models--intfloat--e5-large-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4794.19it/s]
Bert

  ✅ E5 model loaded.
  [after E5 load] VRAM: 2.51 GB allocated | 2.52 GB reserved | 4.00 GB total

[8.2] Converting chunks to Documents (prefix="passage: ") ...
  ✅ 138 Documents ready.
  Sample page_content prefix check: 'passage: **Clinical ...'

[8.3] Ingesting into c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_e5 ...
  Ingesting 138 docs in 5 batches of 32...
  Batch  1/5:  32 docs  [8409dedeea26 … ea9194262a27]
  Batch  2/5:  32 docs  [1fddeefdfb8b … 38f855618fe2]
  Batch  3/5:  32 docs  [0c49aa8d13c4 … 77e6e32d188a]
  Batch  4/5:  32 docs  [e63bd0378895 … dd7540d2be3e]
  Batch  5/5:  10 docs  [24994a09fcdd … b4b0b902c29a]

  ✅ Ingestion complete — 138 documents in c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_e5

[8.4] Verifying db_wound_care_v4_e5 ...

  Total docs   : 138  (expected 138)
  ✅ Count verified: 138

  Docs by source:
            : 108
    ANZBA   :   4
    EWMA    :  12
    ISTAP   

## Cell 9 — Final Cross-Verification (All 3 DBs)

Confirms all three DBs have 138 documents each and that the MedEmbed DB
was not accidentally overwritten. No embedding model is loaded here —
this cell uses only ChromaDB's raw `.get()` (no re-embedding needed).

In [10]:
print("\n" + "=" * 65)
print("  FINAL CROSS-VERIFICATION — All 3 ChromaDB Collections")
print("=" * 65)

import chromadb

def _raw_count(db_path: Path) -> int:
    """Return the document count from a ChromaDB directory without loading an embedding model."""
    client = chromadb.PersistentClient(path=str(db_path))
    cols   = client.list_collections()
    if not cols:
        return 0
    return cols[0].count()


checks = [
    ("R4-A  MedEmbed", DB_MEDEMBED, "db_wound_care_v4       (must NOT be 0 — not rebuilt)"),
    ("R4-B  BGE     ", DB_BGE,      "db_wound_care_v4_bge   (built in Cell 7)"),
    ("R4-C  E5      ", DB_E5,       "db_wound_care_v4_e5    (built in Cell 8)"),
]

all_ok = True
for label, db_path, note in checks:
    if not db_path.exists():
        print(f"  ❌ {label}: DIRECTORY NOT FOUND at {db_path}")
        all_ok = False
        continue
    count = _raw_count(db_path)
    status = "✅" if count == 138 else "❌"
    if count != 138:
        all_ok = False
    print(f"  {status} {label}: {count:>3} docs  |  {note}")

print()
if all_ok:
    print("  ✅ ALL CHECKS PASSED — Ready to run ragas_ablation_R4_embedding.py")
else:
    print("  ❌ ONE OR MORE CHECKS FAILED — investigate before running R4 ablation")

print("\n  DB locations for R4 ablation script config:")
print(f"    R4-A (MedEmbed) : {DB_MEDEMBED}")
print(f"    R4-B (BGE)      : {DB_BGE}")
print(f"    R4-C (E5)       : {DB_E5}")
print("\n  E5 query prefix reminder:")
print("    At search time in R4 ablation, prepend 'query: ' to all E5 sub-queries.")
print("    This is handled automatically in ragas_ablation_R4_embedding.py.")



  FINAL CROSS-VERIFICATION — All 3 ChromaDB Collections
  ✅ R4-A  MedEmbed: 138 docs  |  db_wound_care_v4       (must NOT be 0 — not rebuilt)
  ✅ R4-B  BGE     : 138 docs  |  db_wound_care_v4_bge   (built in Cell 7)
  ✅ R4-C  E5      : 138 docs  |  db_wound_care_v4_e5    (built in Cell 8)

  ✅ ALL CHECKS PASSED — Ready to run ragas_ablation_R4_embedding.py

  DB locations for R4 ablation script config:
    R4-A (MedEmbed) : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4
    R4-B (BGE)      : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_bge
    R4-C (E5)       : c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\db_wound_care_v4_e5

  E5 query prefix reminder:
    At search time in R4 ablation, prepend 'query: ' to all E5 sub-queries.
    This is handled automatically in ragas_ablation_R4_embedding.py.
